# MyDream 전체 시퀀스 모델 비교 파이프라인

이 노트북은 Google Drive에 저장한 원본 수면 JSONL에서 시작해 전처리, 시퀀스 데이터셋 생성, 기준 GRU 학습, 확장 모델 학습 및 비교까지 한 Colab 런타임에서 순서대로 수행합니다.

생성된 데이터셋과 모델 결과는 Google Drive의 `PROFILE_ROOT` 아래에 저장되므로 런타임이 종료되어도 재사용할 수 있습니다.

## 1. 런타임 및 코드 준비

가능하면 GPU 런타임을 선택합니다. 새 Colab 런타임이면 코드 저장소를 복제하고 필요한 패키지를 설치합니다.

In [ ]:
!pip -q install tensorflow pandas matplotlib seaborn scikit-learn joblib

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')

CODE_ROOT = Path('/content/mydream-training-evaluation')
REPOSITORY_URL = 'https://github.com/sfpahsdev-mydream/mydream-training-evaluation.git'
if not (CODE_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(CODE_ROOT)], check=True)

assert (CODE_ROOT / 'parse_sleep_export.py').exists(), CODE_ROOT
print('코드 폴더 준비 완료:', CODE_ROOT)

## 2. 영구 저장 경로 및 기상 정책 설정

`RAW_EXPORT`를 Google Drive에 올린 실제 JSONL 파일 경로로 수정합니다. 아래 기본 설정은 평일 `07:00`, 주말 `09:00` 고정 기상 정책이며, 다른 정책을 평가하려면 이 셀의 값을 변경합니다.

In [ ]:
RAW_EXPORT = Path('/content/drive/MyDrive/mydream_latest/input/mydream_sleep.jsonl')
PROFILE_ROOT = Path('/content/drive/MyDrive/mydream_latest/out/latest_fixed_wake_policy')

WAKE_TIME_POLICY = 'fixed_weekday_weekend'
WEEKDAY_WAKE_TIME = '07:00'
WEEKEND_WAKE_TIME = '09:00'
INCLUDE_TABULAR = False
REUSE_EXISTING_RESULTS = False
THRESHOLDS = [0.4, 0.5, 0.55, 0.6]
REPEAT_SEEDS = [42, 43, 44, 45, 46]

SELECTED_GRU_DIR = PROFILE_ROOT / 'sequence_experiments' / 'gru' / 'gru64_dense32_dropout00'
TABULAR_DIR = PROFILE_ROOT / 'model_tabular_tflite'

assert RAW_EXPORT.exists(), f'원본 JSONL 파일이 없습니다: {RAW_EXPORT}'
PROFILE_ROOT.mkdir(parents=True, exist_ok=True)

print('원본 파일:', RAW_EXPORT)
print('결과 저장 폴더:', PROFILE_ROOT)
print('기상 정책:', WAKE_TIME_POLICY, WEEKDAY_WAKE_TIME, WEEKEND_WAKE_TIME)
print('테이블 모델 포함 여부:', INCLUDE_TABULAR)
print('기존 모델 결과 재사용 여부:', REUSE_EXISTING_RESULTS)
print('반복 평가 seed:', REPEAT_SEEDS)

## 3. 원본 수면 데이터 전처리

원본 JSONL을 파싱해 세션, 단계, 학습 후보, 알람 후보 CSV를 `PROFILE_ROOT`에 생성합니다.

In [ ]:
parse_command = [
    sys.executable,
    str(CODE_ROOT / 'parse_sleep_export.py'),
    '--input', str(RAW_EXPORT),
    '--out-dir', str(PROFILE_ROOT),
    '--wake-time-policy', WAKE_TIME_POLICY,
]
if WAKE_TIME_POLICY == 'fixed_weekday_weekend':
    parse_command += ['--weekday-wake-time', WEEKDAY_WAKE_TIME, '--weekend-wake-time', WEEKEND_WAKE_TIME]

print('실행 명령:', ' '.join(parse_command))
subprocess.run(parse_command, cwd=CODE_ROOT, check=True)

for filename in ['stages.csv', 'training_candidates_1min.csv', 'alarm_candidates_1min.csv']:
    assert (PROFILE_ROOT / filename).exists(), f'전처리 결과가 없습니다: {filename}'

## 4. 시퀀스 데이터셋 생성

학습용 후보와 알람 구간 평가용 후보에서 각각 60분 시퀀스 입력을 생성합니다.

In [ ]:
sequence_jobs = [
    ('training_candidates_1min.csv', PROFILE_ROOT / 'sequence_60m'),
    ('alarm_candidates_1min.csv', PROFILE_ROOT / 'sequence_60m_alarm'),
]
for candidates_file, output_dir in sequence_jobs:
    command = [
        sys.executable,
        str(CODE_ROOT / 'build_sequence_dataset.py'),
        '--input-dir', str(PROFILE_ROOT),
        '--candidates-file', candidates_file,
        '--output-dir', str(output_dir),
    ]
    print('실행 명령:', ' '.join(command))
    subprocess.run(command, cwd=CODE_ROOT, check=True)

for directory in [PROFILE_ROOT / 'sequence_60m', PROFILE_ROOT / 'sequence_60m_alarm']:
    assert (directory / 'sequence_stage_ids.npy').exists(), directory
    assert (directory / 'sequence_metadata.csv').exists(), directory

## 5. 기준 GRU 학습

확장 후보를 비교할 기준 모델인 `gru64_dense32_dropout00`을 학습합니다. 같은 입력과 정책으로 중단된 실행을 이어갈 때만 `REUSE_EXISTING_RESULTS = True`로 설정해 기존 결과를 재사용합니다.

In [ ]:
selected_gru_predictions = SELECTED_GRU_DIR / 'alarm_predictions_long.csv'
if REUSE_EXISTING_RESULTS and selected_gru_predictions.exists():
    print('기존 기준 GRU 결과 재사용:', selected_gru_predictions)
else:
    gru_command = [
        sys.executable,
        str(CODE_ROOT / 'train_sequence_colab.py'),
        '--sequence-dir', str(PROFILE_ROOT / 'sequence_60m'),
        '--predict-sequence-dir', str(PROFILE_ROOT / 'sequence_60m_alarm'),
        '--output-dir', str(SELECTED_GRU_DIR),
        '--model-type', 'gru',
        '--hidden-units', '64',
        '--dense-units', '32',
        '--dropout', '0.0',
    ]
    print('실행 명령:', ' '.join(gru_command))
    subprocess.run(gru_command, cwd=CODE_ROOT, check=True)

assert selected_gru_predictions.exists(), selected_gru_predictions
print('기준 GRU 결과:', SELECTED_GRU_DIR)

## 6. 선택 사항: 테이블 TFLite 모델 학습

`INCLUDE_TABULAR = True`인 경우에만 확장 비교에 포함할 테이블 TFLite 모델을 학습합니다. LightGBM 결과가 아니라 `model_tabular_tflite/alarm_predictions_long.csv`를 생성하는 단계입니다.

In [ ]:
if INCLUDE_TABULAR:
    tabular_predictions = TABULAR_DIR / 'alarm_predictions_long.csv'
    if REUSE_EXISTING_RESULTS and tabular_predictions.exists():
        print('기존 테이블 모델 결과 재사용:', tabular_predictions)
    else:
        tabular_command = [
            sys.executable,
            str(CODE_ROOT / 'train_tabular_tflite_colab.py'),
            '--input-dir', str(PROFILE_ROOT),
            '--output-dir', str(TABULAR_DIR),
            '--float16',
        ]
        print('실행 명령:', ' '.join(tabular_command))
        subprocess.run(tabular_command, cwd=CODE_ROOT, check=True)
    assert tabular_predictions.exists(), tabular_predictions
else:
    print('테이블 모델 비교를 생략합니다.')

## 7. 확장 후보 모델 학습

TCN, Transformer, CNN+GRU를 학습하고 기준 GRU와 함께 표준 비교 결과를 생성합니다. 같은 입력과 정책으로 중단된 실행을 이어갈 때만 기존 후보 결과를 재사용합니다.

In [ ]:
matrix_command = [
    sys.executable,
    str(CODE_ROOT / 'run_sequence_experiment_matrix.py'),
    '--profile-root', str(PROFILE_ROOT),
    '--experiment-set', 'expanded',
    '--comparison-model-dir', str(SELECTED_GRU_DIR),
]
if REUSE_EXISTING_RESULTS:
    matrix_command.append('--skip-existing')
if not INCLUDE_TABULAR:
    matrix_command.append('--no-tabular-model')

print('실행 명령:', ' '.join(matrix_command))
subprocess.run(matrix_command, cwd=CODE_ROOT, check=True)

## 8. 동일 임계값으로 비교 결과 다시 계산

배포 기준 후보 임계값 `0.55`를 포함해 모든 아키텍처를 `0.4`, `0.5`, `0.55`, `0.6`에서 비교합니다.

In [ ]:
comparison_root = PROFILE_ROOT / 'sequence_experiments' / 'expanded'
comparison_output = comparison_root / 'alarm_failure_comparison_selected_thresholds'

model_dirs = [
    SELECTED_GRU_DIR,
    comparison_root / 'tcn64_dense32_dropout00',
    comparison_root / 'transformer64_dense32_dropout10',
    comparison_root / 'cnn32_gru64_dense32_dropout00',
]
if INCLUDE_TABULAR:
    model_dirs.insert(0, TABULAR_DIR)

for model_dir in model_dirs:
    predictions = model_dir / 'alarm_predictions_long.csv'
    assert predictions.exists(), f'모델 예측 결과가 없습니다: {predictions}'

analysis_command = [sys.executable, str(CODE_ROOT / 'analyze_alarm_failures.py')]
for model_dir in model_dirs:
    analysis_command += ['--model-dir', str(model_dir)]
for threshold in THRESHOLDS:
    analysis_command += ['--threshold', str(threshold), '--focus-threshold', str(threshold)]
analysis_command += ['--output-dir', str(comparison_output)]

print('실행 명령:', ' '.join(analysis_command))
subprocess.run(analysis_command, cwd=CODE_ROOT, check=True)
print('비교 결과 폴더:', comparison_output)

## 9. 비교 결과 확인

`strong_fail`은 낮게 유지하면서 `deep_success`와 `success_per_smart`가 높은 모델을 우선합니다.

In [ ]:
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

summary = pd.read_csv(comparison_output / 'model_comparison_summary.csv')
summary = summary.sort_values(['threshold', 'strong_fail', 'deep_success'], ascending=[True, True, False])
display(summary.reset_index(drop=True))

reference = summary[summary['threshold'].eq(0.55)].copy()
reference = reference.sort_values(['strong_fail', 'deep_success', 'success_per_smart'], ascending=[True, False, False])
print('임계값 0.55 비교')
display(reference.reset_index(drop=True))

sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(data=summary, x='strong_fail', y='deep_success', hue='model', style='threshold', s=120, ax=axes[0])
axes[0].set_title('임계값별 알람 성능 균형')
axes[0].set_xlabel('강한 실패 (낮을수록 좋음)')
axes[0].set_ylabel('깊은 수면 성공 (높을수록 좋음)')

plot_data = reference.melt(id_vars=['model', 'threshold'], value_vars=['deep_success', 'strong_fail'], var_name='metric', value_name='count')
sns.barplot(data=plot_data, x='model', y='count', hue='metric', errorbar=None, ax=axes[1])
axes[1].set_title('임계값 0.55에서 모델별 개수')
axes[1].set_xlabel('모델')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 10. 반복 평가 실행

GRU, TCN, Transformer, CNN+GRU를 같은 seed 목록으로 반복 학습합니다. 각 seed와 임계값에서 GRU 대비 개선이 일관되게 유지되는지 집계하고, 테이블 모델이 있으면 동일 정책 비교까지 수행합니다.

In [ ]:
repeat_command = [
    sys.executable,
    str(CODE_ROOT / 'run_repeated_sequence_evaluation.py'),
    '--profile-root', str(PROFILE_ROOT),
]
for threshold in THRESHOLDS:
    repeat_command += ['--threshold', str(threshold)]
for seed in REPEAT_SEEDS:
    repeat_command += ['--seed', str(seed)]
if INCLUDE_TABULAR:
    repeat_command.append('--evaluate-policies')
if REUSE_EXISTING_RESULTS:
    repeat_command.append('--skip-existing')

print('실행 명령:', ' '.join(repeat_command))
subprocess.run(repeat_command, cwd=CODE_ROOT, check=True)
repeated_output = PROFILE_ROOT / 'sequence_experiments' / 'repeated_evaluation'
print('반복 평가 결과 폴더:', repeated_output)

## 11. 반복 평가 집계 확인

`delta_vs_gru_summary.csv`에서 성공 변화는 양수, 강한 실패 변화는 음수일수록 좋습니다. 여러 seed에서 같은 방향으로 유지되는 후보를 우선합니다.

In [ ]:
aggregate_summary = pd.read_csv(repeated_output / 'aggregate_summary.csv')
delta_summary = pd.read_csv(repeated_output / 'delta_vs_gru_summary.csv')
per_seed_summary = pd.read_csv(repeated_output / 'per_seed_summary.csv')
policy_summary_path = repeated_output / 'policy_aggregate_summary.csv'

print('모델별 반복 평가 요약')
display(aggregate_summary)
print('GRU 대비 차이 요약')
display(delta_summary)
display(per_seed_summary.sort_values(['threshold', 'seed', 'strong_fail', 'deep_success'], ascending=[True, True, True, False]))
if policy_summary_path.exists():
    print('모델별 정책 반복 평가 요약')
    display(pd.read_csv(policy_summary_path))

## 12. Android 전달용 TFLite 산출물 생성

반복 평가가 끝난 후보 모델을 Android 앱에 넣을 수 있도록 float32/float16 TFLite, scaler, manifest, 평가 요약을 `android_assets/` 아래에 모읍니다. GRU는 TFLite 변환 안정성을 위해 unrolled GRU로 재빌드하고, TCN/Transformer/CNN+GRU는 저장된 Keras 모델을 직접 변환합니다.

In [ ]:
import json
import shutil

ANDROID_ASSETS_DIR = PROFILE_ROOT / 'android_assets'
ANDROID_TFLITE_DIR = ANDROID_ASSETS_DIR / 'tflite'
ANDROID_SUMMARY_DIR = ANDROID_ASSETS_DIR / 'evaluation_summary'
ANDROID_ASSETS_DIR.mkdir(parents=True, exist_ok=True)
ANDROID_TFLITE_DIR.mkdir(parents=True, exist_ok=True)
ANDROID_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

android_sequence_models = [
    ('gru64_dense32_dropout00', SELECTED_GRU_DIR, False),
    ('tcn64_dense32_dropout00', comparison_root / 'tcn64_dense32_dropout00', True),
    ('transformer64_dense32_dropout10', comparison_root / 'transformer64_dense32_dropout10', True),
    ('cnn32_gru64_dense32_dropout00', comparison_root / 'cnn32_gru64_dense32_dropout00', False),
]

android_manifests = []
for model_name, model_dir, direct_convert in android_sequence_models:
    keras_model = model_dir / 'sequence_model.keras'
    assert keras_model.exists(), f'Missing Keras model for Android export: {keras_model}'
    output_dir = ANDROID_TFLITE_DIR / model_name
    if output_dir.exists() and not REUSE_EXISTING_RESULTS:
        shutil.rmtree(output_dir)
    command = [
        sys.executable,
        str(CODE_ROOT / 'convert_sequence_model_tflite.py'),
        '--model-dir', str(model_dir),
        '--output-dir', str(output_dir),
        '--sequence-dir', str(PROFILE_ROOT / 'sequence_60m_alarm'),
        '--float16',
    ]
    if direct_convert:
        command.append('--no-rebuild-unrolled-gru')
    print('실행 명령:', ' '.join(command))
    subprocess.run(command, cwd=CODE_ROOT, check=True)
    manifest_path = output_dir / 'tflite_manifest.json'
    assert manifest_path.exists(), manifest_path
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    manifest['model_name'] = model_name
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
    android_manifests.append(manifest)

if INCLUDE_TABULAR and TABULAR_DIR.exists():
    tabular_target = ANDROID_ASSETS_DIR / 'tabular_model_tflite'
    if tabular_target.exists():
        shutil.rmtree(tabular_target)
    shutil.copytree(TABULAR_DIR, tabular_target)

summary_files = [
    comparison_output / 'model_comparison_summary.csv',
    repeated_output / 'aggregate_summary.csv',
    repeated_output / 'delta_vs_gru_summary.csv',
    repeated_output / 'per_seed_summary.csv',
    repeated_output / 'policy_aggregate_summary.csv',
]
for source in summary_files:
    if source.exists():
        shutil.copy2(source, ANDROID_SUMMARY_DIR / source.name)

android_package_manifest = {
    'profile_root': str(PROFILE_ROOT),
    'thresholds': THRESHOLDS,
    'repeat_seeds': REPEAT_SEEDS,
    'sequence_models': android_manifests,
    'tabular_included': bool(INCLUDE_TABULAR and TABULAR_DIR.exists()),
    'recommended_android_benchmark_budget': {
        'cadence': 'once_per_minute',
        'hard_p95_ms': 1000,
        'preferred_p95_ms': 500,
    },
}
(ANDROID_ASSETS_DIR / 'android_assets_manifest.json').write_text(
    json.dumps(android_package_manifest, indent=2, sort_keys=True),
    encoding='utf-8',
)

android_archive = shutil.make_archive(str(ANDROID_ASSETS_DIR), 'zip', root_dir=ANDROID_ASSETS_DIR)
print('Android assets folder:', ANDROID_ASSETS_DIR)
print('Android assets archive:', android_archive)


## 13. 비교 파일 압축

단일 비교 결과와 반복 평가 집계는 Drive에 이미 저장됩니다. 필요한 경우 두 결과 폴더를 ZIP 파일로 함께 저장합니다.

In [ ]:
import shutil

comparison_archive = shutil.make_archive(str(comparison_output), 'zip', root_dir=comparison_output)
repeated_archive = shutil.make_archive(str(repeated_output), 'zip', root_dir=repeated_output)
print('단일 비교 압축 파일:', comparison_archive)
print('반복 평가 압축 파일:', repeated_archive)
if 'android_archive' in globals():
    print('Android assets 압축 파일:', android_archive)
